RoBERTa Training with Strong Regularization

Key techniques:
- Increased dropout (attn=0.2, hidden=0.3)
- Weight decay=0.05
- Focal Loss (gamma=2.0) with class weights
- Label smoothing=0.15
- Early stopping (patience=3)

Inputs:
- models/roberta-brexit-dapt (base)

Outputs:
- models/roberta-dapt-regularized/
- results/regularized_training_metrics.json


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import json
import pickle
import numpy as np
import pandas as pd
import torch
import warnings
from tqdm.auto import tqdm

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("✅ Libraries loaded")
print(f"Device: {device}")


✅ Libraries loaded
Device: cuda


## 1) Load augmented data and prepare splits


In [2]:
print("Loading augmented data...")
df = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])

print(f"✅ Loaded {len(df):,} samples")
print("Split distribution:\n", df['split'].value_counts())
print("Train class distribution:\n", df[df['split']=='train']['frame_label'].value_counts().sort_index())

train_df = df[df['split']=='train'].copy()
val_df = df[df['split']=='validation'].copy()
test_df = df[df['split']=='test'].copy()

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(df['frame_label'])
labels = list(label_encoder.classes_)
num_labels = len(labels)

train_df['label'] = label_encoder.transform(train_df['frame_label'])
val_df['label'] = label_encoder.transform(val_df['frame_label'])
test_df['label'] = label_encoder.transform(test_df['frame_label'])

os.makedirs('data', exist_ok=True)
with open('data/roberta_label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("✅ Saved label encoder")


Loading augmented data...
✅ Loaded 5,637 samples
Split distribution:
 split
train         4606
validation     683
test           348
Name: count, dtype: int64
Train class distribution:
 frame_label
Conflict         931
Economic         646
Human Impact     522
Moral Value      942
None             633
Powerlessness    932
Name: count, dtype: int64
✅ Saved label encoder


## 2) Tokenize and build HF datasets


In [3]:
print("Loading tokenizer from models/roberta-brexit-dapt ...")
tokenizer = RobertaTokenizer.from_pretrained('models/roberta-brexit-dapt')
print("✅ Tokenizer loaded")


def tokenize_function(examples):
    return tokenizer(
        examples['chunk_text'],
        truncation=True,
        max_length=384,
        padding='max_length'
    )

train_dataset = Dataset.from_pandas(train_df[['chunk_text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['chunk_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['chunk_text', 'label']])

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])

train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')
print("✅ Tokenization complete")


Loading tokenizer from models/roberta-brexit-dapt ...
✅ Tokenizer loaded
Tokenizing...


Map:   0%|          | 0/4606 [00:00<?, ? examples/s]

Map:   0%|          | 0/683 [00:00<?, ? examples/s]

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

✅ Tokenization complete


## 3) Define Focal Loss and custom Trainer


In [4]:
import torch.nn as nn
import torch.nn.functional as F

# NO CLASS WEIGHTS
# Reasoning:
#   1. Training data imbalance is intentional (based on baseline performance)
#   2. Imbalance ratio 1.8:1 (522-942 samples) is MILD for deep learning
#   3. Experiments show ANY class weighting causes model collapse
#   4. Standard CE Loss will learn all classes naturally with 4,606 samples
print("✅ Using standard Cross Entropy (NO class weights)")

class StandardCETrainer(Trainer):
    """Custom Trainer with standard Cross Entropy Loss - no modifications"""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Standard unweighted Cross Entropy
        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("✅ Standard CE Trainer ready (no weights)")

✅ Using standard Cross Entropy (NO class weights)
✅ Standard CE Trainer ready (no weights)


## 4) Load base model with increased dropout


In [5]:
print("Loading base model from models/roberta-brexit-dapt ...")
model = RobertaForSequenceClassification.from_pretrained(
    'models/roberta-brexit-dapt',
    num_labels=num_labels,
    attention_probs_dropout_prob=0.1,   # Reduced from 0.2
    hidden_dropout_prob=0.15,            # Reduced from 0.3
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("✅ Model loaded")
print(f"   Dropout: attention={0.1}, hidden={0.15} (reduced for better minority class learning)")

Loading base model from models/roberta-brexit-dapt ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at models/roberta-brexit-dapt and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded
   Dropout: attention=0.1, hidden=0.15 (reduced for better minority class learning)


## 5) Metrics and Training Arguments


In [6]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average='macro')
    precision, recall, f1, support = precision_recall_fscore_support(labels_np, preds, average=None, zero_division=0)
    metrics = {'accuracy': float(acc), 'f1_macro': float(f1_macro)}
    for idx, name in enumerate(labels):
        metrics[f'f1_{name}'] = float(f1[idx])
    return metrics

# DIFFERENTIAL LEARNING RATES
# Critical fix: Fresh classifier needs higher LR than pretrained encoder
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if "classifier" not in n],
        "lr": 2e-5,  # Encoder: low LR (preserve DAPT)
    },
    {
        "params": [p for n, p in model.named_parameters() if "classifier" in n],
        "lr": 5e-4,  # Classifier: 25x higher (AGGRESSIVE for random init)
    },
]

print("✅ Differential learning rates configured:")
print(f"   Encoder (DAPT weights): 2e-5")
print(f"   Classifier (random init): 5e-4 (25x higher - AGGRESSIVE)")

training_args = TrainingArguments(
    output_dir='models/roberta-dapt-regularized',
    num_train_epochs=10,
    learning_rate=2e-5,  # Base (overridden by optimizer groups)
    lr_scheduler_type='cosine',
    warmup_ratio=0.15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    max_grad_norm=5.0,  # Increased: allow larger gradients for fresh classifier
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    label_smoothing_factor=0.0,  # REMOVED: interferes with confident learning
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to='none',
    seed=42,
)
print("✅ Training args ready")

✅ Differential learning rates configured:
   Encoder (DAPT weights): 2e-5
   Classifier (random init): 5e-4 (25x higher - AGGRESSIVE)
✅ Training args ready


## 6) Train and evaluate


In [7]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# Create optimizer with differential learning rates
optimizer = AdamW(optimizer_grouped_parameters, lr=2e-5, weight_decay=0.01)

# Calculate training steps for scheduler
num_training_steps = len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs
num_warmup_steps = int(num_training_steps * training_args.warmup_ratio)

# Create cosine scheduler
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f"Training schedule: {num_training_steps} steps, {num_warmup_steps} warmup")

trainer = StandardCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    optimizers=(optimizer, scheduler)  # Custom optimizer with differential LRs
)

print("\n🚀 Starting training with AGGRESSIVE differential learning rates...")
print("Configuration:")
print(f"  - Loss: Standard Cross Entropy (unweighted)")
print(f"  - Encoder LR: 2e-5 (preserve DAPT domain knowledge)")
print(f"  - Classifier LR: 5e-4 (AGGRESSIVE - 25x encoder)")
print(f"  - Dropout: 0.1/0.15 (moderate)")
print(f"  - Weight decay: 0.01")
print(f"  - Label smoothing: 0.0 (DISABLED - allow confident learning)")
print(f"  - Gradient clipping: 5.0 (relaxed for fresh classifier)")
print(f"  - Training samples: {len(train_dataset):,}")
print(f"  - Validation samples: {len(val_dataset):,}")
print("="*80)

train_result = trainer.train()
print("✅ Training complete")

print("\nEvaluating on validation set...")
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(val_results)

print("\nEvaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(test_results)

# Save model and metrics
os.makedirs('models/roberta-dapt-regularized', exist_ok=True)
trainer.save_model('models/roberta-dapt-regularized')
tokenizer.save_pretrained('models/roberta-dapt-regularized')

os.makedirs('results', exist_ok=True)
with open('results/regularized_training_metrics.json', 'w') as f:
    json.dump({
        'validation': val_results,
        'test': test_results,
        'classes': labels
    }, f, indent=2)
print("✅ Saved model and metrics")

Training schedule: 1430 steps, 214 warmup

🚀 Starting training with AGGRESSIVE differential learning rates...
Configuration:
  - Loss: Standard Cross Entropy (unweighted)
  - Encoder LR: 2e-5 (preserve DAPT domain knowledge)
  - Classifier LR: 5e-4 (AGGRESSIVE - 25x encoder)
  - Dropout: 0.1/0.15 (moderate)
  - Weight decay: 0.01
  - Label smoothing: 0.0 (DISABLED - allow confident learning)
  - Gradient clipping: 5.0 (relaxed for fresh classifier)
  - Training samples: 4,606
  - Validation samples: 683


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
50,1.783700,1.797224,0.166911,0.047679,0.000000,0.000000,0.000000,0.286073,0.000000,0.000000
100,1.626900,1.204458,0.541728,0.543603,0.469484,0.692308,0.592965,0.426540,0.597561,0.482759
150,0.989800,0.930196,0.651537,0.653386,0.629213,0.783333,0.673077,0.546256,0.715789,0.572650
200,0.734800,0.961152,0.670571,0.669355,0.595349,0.785992,0.738318,0.593496,0.775701,0.527273
250,0.661100,1.046961,0.655930,0.657215,0.534091,0.758621,0.735294,0.577617,0.751323,0.586345
300,0.522200,1.225360,0.636896,0.634595,0.575916,0.716216,0.669856,0.592593,0.674556,0.578431
350,0.296400,1.217788,0.689605,0.689321,0.620690,0.816000,0.775330,0.600000,0.734694,0.589212
400,0.221700,1.432338,0.661786,0.662260,0.560847,0.781818,0.731707,0.617761,0.710000,0.571429
450,0.186000,1.461195,0.663250,0.661637,0.625551,0.834711,0.732510,0.502370,0.692308,0.582375
500,0.100500,1.721223,0.638360,0.638737,0.552083,0.800000,0.723735,0.497561,0.717949,0.541096


✅ Training complete

Evaluating on validation set...


{'eval_loss': 1.2177881002426147, 'eval_accuracy': 0.6896046852122987, 'eval_f1_macro': 0.6893209245760777, 'eval_f1_Conflict': 0.6206896551724138, 'eval_f1_Economic': 0.816, 'eval_f1_Human Impact': 0.775330396475771, 'eval_f1_Moral Value': 0.6, 'eval_f1_None': 0.7346938775510204, 'eval_f1_Powerlessness': 0.5892116182572614, 'eval_runtime': 25.3188, 'eval_samples_per_second': 26.976, 'eval_steps_per_second': 0.869, 'epoch': 3.4722222222222223}

Evaluating on test set...
{'eval_loss': 1.1083229780197144, 'eval_accuracy': 0.6839080459770115, 'eval_f1_macro': 0.6838933281621028, 'eval_f1_Conflict': 0.6666666666666666, 'eval_f1_Economic': 0.7666666666666667, 'eval_f1_Human Impact': 0.743801652892562, 'eval_f1_Moral Value': 0.6349206349206349, 'eval_f1_None': 0.7, 'eval_f1_Powerlessness': 0.591304347826087, 'eval_runtime': 12.8254, 'eval_samples_per_second': 27.134, 'eval_steps_per_second': 0.858, 'epoch': 3.4722222222222223}
✅ Saved model and metrics
